# 07 Analysis Preflight

## Overview
This notebook is the analysis-readiness gate between canonical data production and downstream statistics, visualization, and enrichment workflows.

## Scientific Role
1. Resolve the canonical all-cells AnnData artifact used by downstream notebooks.
2. Validate the required observation schema for counts, DGE, plotting, and GSEA workflows.
3. Confirm basic group and sample balance before any inferential analysis begins.
4. Freeze a reproducible handoff contract for notebooks 08-14.

## Upstream Dependency
- `adatas/brain_allcells_allgenes.h5ad` produced from `02_Mouse_PrepForComparison.ipynb`

## Outputs
- Human-readable preflight QA summaries
- Structured run metadata describing the validated canonical input
- Explicit handoff checks for downstream notebooks 08-14

## Standards
- Schema-first validation (`sample`, `group`, `class`, `subclass`, `supertype`, `cluster`)
- No branch-specific statistics or exploratory hypothesis testing in this notebook
- Stable pathing and deterministic checks for reproducibility
- Clear pass/fail style summaries for downstream handoff decisions

# Table of Contents

1. [Overview](#overview)
2. [Configuration and Environment](#configuration-and-environment)
3. [Input Resolution and Schema Validation](#input-resolution-and-schema-validation)
4. [Group and Sample QA](#group-and-sample-qa)
5. [Handoff](#handoff)

## Configuration and Environment

This section initializes imports, paths, and deterministic notebook behavior before any data loading.

## Preflight Acceptance Criteria

A successful preflight run means:

- The canonical AnnData input resolves without fallback ambiguity
- Required observation columns are present and non-empty
- Group and sample membership can be summarized cleanly
- Downstream output directories exist and are ready for notebooks 08-14
- The handoff summary at the end reports `ready = True`

In [ ]:
# Intro is now provided in the markdown cell at the top of this notebook.
# This cell is intentionally left minimal to keep execution flow clean.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Final, Sequence, TypedDict
import warnings

import anndata as ad
import pandas as pd
import scanpy as sc
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False)

In [ ]:
@dataclass(frozen=True)
class Notebook07Config:
    """Centralized filesystem configuration for preflight checks."""

    analysis_root: Path
    results_root: Path
    dge_output_dir: Path
    count_output_dir: Path


NOTEBOOK_DIR: Final[Path] = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "Notebooks":
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "Notebooks").resolve()
ANALYSIS_ROOT: Final[Path] = NOTEBOOK_DIR.parent.resolve()

CONFIG = Notebook07Config(
    analysis_root=ANALYSIS_ROOT,
    results_root=ANALYSIS_ROOT / "Results",
    dge_output_dir=ANALYSIS_ROOT / "Results" / "DGE",
    count_output_dir=ANALYSIS_ROOT / "Results" / "Celltype_counts",
)


def ensure_directories_exist(paths: Sequence[Path]) -> None:
    """Create required output directories if they do not already exist."""

    for path in paths:
        path.mkdir(parents=True, exist_ok=True)


ensure_directories_exist(
    [
        CONFIG.results_root,
        CONFIG.dge_output_dir,
        CONFIG.count_output_dir,
    ]
)

print(f"Analysis root: {CONFIG.analysis_root}")
print(f"Results root: {CONFIG.results_root}")

## Input Resolution and Schema Validation

This section resolves the canonical AnnData input and enforces required metadata columns before downstream analyses.

In [ ]:
REQUIRED_OBS_COLUMNS: Final[tuple[str, ...]] = (
    "class",
    "subclass",
    "supertype",
    "cluster",
    "group",
    "sample",
)

GROUP_ALIAS_MAP: Final[dict[str, str]] = {
    "OG": "Infected",
    "Mock": "Naive",
}


class PreflightRunMetadata(TypedDict):
    """Typed metadata record describing a validated preflight input."""

    input_path: str
    n_cells: int
    n_genes: int
    required_obs: list[str]
    group_counts: dict[str, int]
    sample_counts: dict[str, int]


def resolve_first_existing_path(candidates: Sequence[Path]) -> Path:
    """Return the first existing path from a precedence-ordered list."""

    for candidate in candidates:
        if candidate.exists():
            return candidate

    formatted = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"No valid AnnData input found. Tried:\n{formatted}")


def maybe_attach_supertype_from_mapping(adata: ad.AnnData, mapping_csv: Path) -> None:
    """Backfill supertype metadata from mapping_output_nb01.csv when available."""

    if "supertype" in adata.obs.columns and adata.obs["supertype"].notna().all():
        return
    if "supertype_name" in adata.obs.columns:
        adata.obs["supertype"] = adata.obs["supertype_name"].astype(str)
        return
    if not mapping_csv.exists():
        return

    mapping_df = pd.read_csv(mapping_csv, comment="#", usecols=["cell_id", "supertype_name"])
    mapping_df = mapping_df.dropna(subset=["cell_id", "supertype_name"]).drop_duplicates("cell_id")
    mapping_series = mapping_df.set_index("cell_id")["supertype_name"].astype(str)

    aligned = mapping_series.reindex(adata.obs_names)
    if "supertype" not in adata.obs.columns:
        adata.obs["supertype"] = aligned.astype("object")
    else:
        missing_mask = adata.obs["supertype"].isna() | (adata.obs["supertype"].astype(str).str.len() == 0)
        adata.obs.loc[missing_mask, "supertype"] = aligned.loc[missing_mask]


def normalize_obs_columns(adata: ad.AnnData, mapping_csv: Path) -> None:
    """Normalize observation metadata columns in-place to the canonical schema."""

    rename_candidates: dict[str, list[str]] = {
        "sample": ["sample", "Sample"],
        "group": ["group", "treatment", "Study_Designation", "Condition"],
        "class": ["class", "class_name", "class_broad"],
        "subclass": ["subclass", "subclass_name"],
        "supertype": ["supertype", "supertype_name"],
        "cluster": ["cluster", "cluster_name", "cluster_alias"],
    }

    for target, options in rename_candidates.items():
        if target in adata.obs.columns:
            continue
        source = next((col for col in options if col in adata.obs.columns), None)
        if source is not None:
            adata.obs[target] = adata.obs[source].astype(str)

    maybe_attach_supertype_from_mapping(adata, mapping_csv)

    if "group" in adata.obs.columns:
        adata.obs["group"] = adata.obs["group"].astype(str).replace(GROUP_ALIAS_MAP)
    if "sample" in adata.obs.columns:
        adata.obs["sample"] = adata.obs["sample"].astype(str)


def validate_required_obs_columns(
    adata: ad.AnnData,
    required_columns: Sequence[str],
) -> None:
    """Raise when required observation metadata columns are absent."""

    missing = [column for column in required_columns if column not in adata.obs.columns]
    if missing:
        raise KeyError(f"Missing required metadata columns: {missing}")

    empty_columns = []
    for column in required_columns:
        values = adata.obs[column]
        if values.notna().sum() == 0:
            empty_columns.append(column)
    if empty_columns:
        raise KeyError(f"Required metadata columns are present but empty: {empty_columns}")


def summarize_group_sample_balance(adata: ad.AnnData) -> pd.DataFrame:
    """Return sample-level cell counts split by group for quick QA."""

    return (
        adata.obs[["sample", "group"]]
        .value_counts()
        .rename("n_cells")
        .reset_index()
        .sort_values(["group", "sample"])
        .reset_index(drop=True)
    )


def counts_to_str_int_dict(series: pd.Series) -> dict[str, int]:
    """Convert pandas count series to a plain string-keyed integer dictionary."""

    return {str(index): int(value) for index, value in series.items()}


def build_run_metadata(
    adata: ad.AnnData,
    adata_path: Path,
    required_obs_columns: Sequence[str],
) -> PreflightRunMetadata:
    """Build structured run metadata used for downstream handoff reporting."""

    return {
        "input_path": str(adata_path),
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "required_obs": list(required_obs_columns),
        "group_counts": counts_to_str_int_dict(adata.obs["group"].value_counts()),
        "sample_counts": counts_to_str_int_dict(adata.obs["sample"].value_counts()),
    }


adata_candidates: list[Path] = [
    CONFIG.analysis_root / "adatas" / "brain_allcells_allgenes.h5ad",
    CONFIG.analysis_root / "combined_filtered_pruning.h5ad",
]

adata_path = resolve_first_existing_path(adata_candidates)
adata_combined = sc.read_h5ad(adata_path)

mapping_candidates = [
    CONFIG.analysis_root / "Mapping" / "mapping_output_nb01.csv",
    CONFIG.analysis_root / "concat" / "Mapping" / "mapping_output_nb01.csv",
]
mapping_csv = resolve_first_existing_path(mapping_candidates)

normalize_obs_columns(adata_combined, mapping_csv)
validate_required_obs_columns(adata_combined, REQUIRED_OBS_COLUMNS)

run_metadata = build_run_metadata(
    adata=adata_combined,
    adata_path=adata_path,
    required_obs_columns=REQUIRED_OBS_COLUMNS,
)

print(f"Using AnnData: {adata_path}")
print(f"Mapping CSV: {mapping_csv}")
print(f"Loaded {adata_combined.n_obs} cells x {adata_combined.n_vars} genes")
print("Group counts:")
display(adata_combined.obs["group"].value_counts().to_frame("n_cells"))
print("Sample balance preview:")
display(summarize_group_sample_balance(adata_combined).head(12))

## Group and Sample QA

This section provides quick, human-readable checks for group/sample balance and metadata uniqueness.

In [ ]:
print("AnnData summary:")
print(adata_combined)

In [ ]:
obs_uniqueness = (
    adata_combined.obs.nunique()
    .sort_values(ascending=False)
    .rename("n_unique_values")
    .to_frame()
)

print("Unique value counts per metadata column:")
display(obs_uniqueness)

## Handoff

Preflight is complete once all checks above run successfully and the handoff summary reports `ready = True`.

Next steps:
- Run notebook `08_Celltype_Counts_and_SigDiff.ipynb` for counts, significance testing, hierarchy export, and focused subtype plotting
- Run notebook `09_DGE_Run_Once.ipynb` for one-time DGE generation
- Run notebook `10_DGE_Visualization.ipynb` for DGE visualization and selective plotting
- Run notebook `11_GSEA_from_DGE.ipynb` for compute-only GSEA from saved DGE outputs
- Run notebook `12_GSEA_Visualization_and_Custom_Lists.ipynb` for GSEA visualization and custom pathway views
- Run notebook `13_C1qa_Microglia.ipynb` for microglia-specific follow-up
- Run notebook `14_Multiomic_Pseudobulk_RPPA.ipynb` for multiomic pseudobulk integration

Notebook 07 should remain a boundary notebook: it validates the canonical input and handoff contract, but does not perform downstream inferential analyses itself.

In [ ]:
preflight_summary = pd.DataFrame(
    [
        {
            "check": "Canonical input exists",
            "value": adata_path.exists(),
            "detail": str(adata_path),
        },
        {
            "check": "Required obs columns present",
            "value": all(col in adata_combined.obs.columns for col in REQUIRED_OBS_COLUMNS),
            "detail": ", ".join(REQUIRED_OBS_COLUMNS),
        },
        {
            "check": "Counts layer present",
            "value": "counts" in adata_combined.layers,
            "detail": f"layers={list(adata_combined.layers.keys())}",
        },
        {
            "check": "At least 2 groups present",
            "value": adata_combined.obs["group"].nunique() >= 2,
            "detail": f"n_groups={adata_combined.obs['group'].nunique()}",
        },
        {
            "check": "At least 2 samples present",
            "value": adata_combined.obs["sample"].nunique() >= 2,
            "detail": f"n_samples={adata_combined.obs['sample'].nunique()}",
        },
        {
            "check": "Results directory ready",
            "value": CONFIG.results_root.exists(),
            "detail": str(CONFIG.results_root),
        },
        {
            "check": "Cell-count output directory ready",
            "value": CONFIG.count_output_dir.exists(),
            "detail": str(CONFIG.count_output_dir),
        },
        {
            "check": "DGE output directory ready",
            "value": CONFIG.dge_output_dir.exists(),
            "detail": str(CONFIG.dge_output_dir),
        },
    ]
)

handoff_summary = {
    "ready": bool(preflight_summary["value"].all()),
    "input_path": str(adata_path),
    "n_cells": int(adata_combined.n_obs),
    "n_genes": int(adata_combined.n_vars),
    "n_groups": int(adata_combined.obs["group"].nunique()),
    "n_samples": int(adata_combined.obs["sample"].nunique()),
    "required_obs_columns": list(REQUIRED_OBS_COLUMNS),
    "downstream_notebooks": [
        "08_Celltype_Counts_and_SigDiff.ipynb",
        "09_DGE_Run_Once.ipynb",
        "10_DGE_Visualization.ipynb",
        "11_GSEA_from_DGE.ipynb",
        "12_GSEA_Visualization_and_Custom_Lists.ipynb",
        "13_C1qa_Microglia.ipynb",
        "14_Multiomic_Pseudobulk_RPPA.ipynb",
    ],
}

print("Preflight acceptance summary:")
display(preflight_summary)

print("Handoff contract:")
display(pd.DataFrame([handoff_summary]))